From the `Bamboo_setup` directory, run the following command:
```bash
jupyter notebook --no-browser --port=8888
```
Copy the URL starting with `http://localhost:8888/`.
When picking the kernel for this notebook, click **Existing Jupyter Server** and paste the URL.
Name your server 'localhost'.
From localhost, select the 'Python 3' kernel.

In [ ]:
import sys
from pathlib import Path
BAMBOO_SETUP = Path.cwd()
NN_POSTPROCESSING = BAMBOO_SETUP / 'src' / 'post_processing' / 'NN'
NOTEBOOKS = NN_POSTPROCESSING / 'notebooks'
sys.path.append(str((BAMBOO_SETUP/'src').resolve()))
import pandas as pd
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 1000)  # Set a larger width to fit the editor window
# pd.set_option('display.max_colwidth', None)  # Allow columns to be fully displayed
%load_ext autoreload

### If any changes are made to the imported modules in this notebook, you must run this cell first to load the new changes (this will reload all modules)

In [7]:
%autoreload 2

# Set up a Data Handler (to load and preprocess all data)

In [ ]:
from post_processing.NN.DataHandler import DataHandler
datahandler = DataHandler(
    workdir=Path('/eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_1013/Reco'),
    tree_name='SL_res_2b_x',
    total_inputs= NN_POSTPROCESSING / 'input/vars40.txt'
)
total_df = datahandler.load_data()
total_df = datahandler.fix_any_mismatch(total_df)
total_df = datahandler.preprocess_data(total_df)
# datahandler.data_quality_summary(total_df)

# Set up your model config

In [9]:
from post_processing.NN.utils import ModelConfig
model_config = ModelConfig(
    name='multi_HH_ttbar_tW',
    type='multi',
    categorization={"HH": ["HH_bbWW"], "ttbar": ["ttbar"], "tW": ["tW"]},
    training_weight_sf={"HH_bbWW": 1.0, "ttbar": 8.0, "tW": 4.0},
    input_vars='All',
    architecture_in_yml=True,
    residual_network=False,
    hiddenlayers=[
        {"type": 'Dense', "units": 16, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4}
        # {"type": 'Dense', "units": 16, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4},
        # {"type": 'Dense', "units": 16, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4}
    ],
    outputlayers=[
        {"type": 'Dense', "units": 3, "kernel_initializer": 'normal', "activation": 'softmax', "act_regularizer": {'l2': 1e-4}, "name": 'output'}
    ],
    compiler={"optimizer": 'adam', "lr": 0.001, "loss": 'categorical_crossentropy'},
    fit={"batch_size": 1024, "epochs": 2, "validation_split": 0.25}
)

# Run DNN

In [ ]:
from post_processing.NN.DNNModel import DNNModel
DNN = DNNModel(model_config=model_config, modeldir= NOTEBOOKS/'model_custom_trial2')
DNN.Run(total_df, fixed_random_seed=True)